In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import os
import torch
import torchvision
import cv2
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from tqdm import tqdm

In [29]:
PROJECT_PATH = "/content/drive/MyDrive/Welding-Defect-Detection"

MODEL_PATH = f"{PROJECT_PATH}/models/weld_final.pth"

DATASET_PATH = f"{PROJECT_PATH}/The Welding Defect Dataset - v2"

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cuda
Tesla T4


In [31]:
class WeldDataset(Dataset):

    def __init__(self, img_dir, label_dir):

        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = os.listdir(img_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.label_dir, os.path.splitext(img_name)[0] + ".txt")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = cv2.resize(image,(640,640))

        h,w = 640,640

        boxes=[]
        labels=[]

        if os.path.exists(label_path):

            with open(label_path) as f:

                for line in f.readlines():

                    c,x,y,bw,bh = map(float,line.split())

                    xmin=max(0,(x-bw/2)*w)
                    ymin=max(0,(y-bh/2)*h)
                    xmax=min(w,(x+bw/2)*w)
                    ymax=min(h,(y+bh/2)*h)

                    boxes.append([xmin,ymin,xmax,ymax])

                    labels.append(int(c)+1)

        boxes=torch.tensor(boxes,dtype=torch.float32)
        labels=torch.tensor(labels,dtype=torch.int64)

        image=torch.tensor(image).permute(2,0,1).float()/255.0

        target={}
        target["boxes"]=boxes
        target["labels"]=labels

        return image,target

In [32]:
def collate_fn(batch):
    return tuple(zip(*batch))


img_dir = os.path.join(DATASET_PATH,"test","images")
label_dir = os.path.join(DATASET_PATH,"test","labels")

test_dataset = WeldDataset(img_dir,label_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

print("Test images:",len(test_dataset))

Test images: 126


In [33]:
num_classes = 4

model = torchvision.models.detection.retinanet_resnet50_fpn(weights=None)

num_anchors = model.head.classification_head.num_anchors
in_channels = model.backbone.out_channels

model.head.classification_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(
    in_channels,
    num_anchors,
    num_classes
)

model.load_state_dict(torch.load(MODEL_PATH,map_location=device))

model.to(device)
model.eval()

print("Model loaded")

Model loaded


In [34]:
TP = 0
FP = 0
FN = 0

IOU_THRESHOLD = 0.5

with torch.no_grad():

    for images,targets in tqdm(test_loader):

        images=[img.to(device) for img in images]

        outputs=model(images)

        for pred,target in zip(outputs,targets):

            pred_boxes=pred["boxes"].cpu()
            pred_scores=pred["scores"].cpu()

            gt_boxes=target["boxes"]

            pred_boxes=pred_boxes[pred_scores>0.3]

            if len(pred_boxes)==0 and len(gt_boxes)==0:
                continue

            if len(pred_boxes)==0:
                FN+=len(gt_boxes)
                continue

            if len(gt_boxes)==0:
                FP+=len(pred_boxes)
                continue

            ious=box_iou(pred_boxes,gt_boxes)

            max_iou,_=ious.max(dim=1)

            TP+=(max_iou>IOU_THRESHOLD).sum().item()
            FP+=(max_iou<=IOU_THRESHOLD).sum().item()

100%|██████████| 32/32 [00:11<00:00,  2.83it/s]


In [35]:
precision = TP/(TP+FP+1e-6)
recall = TP/(TP+FN+1e-6)
f1 = 2*(precision*recall)/(precision+recall+1e-6)

print("\nEvaluation Results")
print("-----------------------")

print("True Positives:",TP)
print("False Positives:",FP)
print("False Negatives:",FN)

print("\nPrecision:",round(precision,4))
print("Recall:",round(recall,4))
print("F1 Score:",round(f1,4))


Evaluation Results
-----------------------
True Positives: 281
False Positives: 95
False Negatives: 22

Precision: 0.7473
Recall: 0.9274
F1 Score: 0.8277
